In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, RadioButtons, HBox, VBox, Layout, HTML, GridBox
from IPython.display import display

# ============================================================
# CSS FOR HORIZONTAL RADIO BUTTONS
# ============================================================

display(HTML("""
<style>
.horizontal-radio .widget-radio-box {
    display: flex !important;
    flex-direction: row !important;
    flex-wrap: nowrap !important;
    align-items: center !important;
    gap: 18px !important;
}
.horizontal-radio .widget-radio-box label {
    margin: 0 !important;
    white-space: nowrap !important;
}
.horizontal-radio > label {
    display: none !important;
}
</style>
"""))

# ============================================================
# INTERACTIVE FUNCTION
# ============================================================

def plot_psd_transformation(input_type='White noise', system_type='Low-pass', rho=0.85, alpha=0.80, center=0.45):

    # --------------------------------------------------------
    # FREQUENCY AXIS
    # --------------------------------------------------------

    omega = np.linspace(
        -np.pi,
        np.pi,
        2048
    )

    # ========================================================
    # INPUT PSD
    # ========================================================

    if input_type == 'White noise':

        Sxx = np.ones_like(omega)

        input_name = 'White Noise'

    else:

        # AR(1):
        # Sxx(w) proportional to 1 / |1 - rho e^(-jw)|^2

        denominator = (
            1.0
            + rho**2
            - 2.0 * rho * np.cos(omega)
        )

        Sxx = (
            1.0 - rho**2
        ) / denominator

        input_name = f'Colored AR(1), ρ = {rho:.2f}'

    # --------------------------------------------------------
    # NORMALIZE INPUT PSD
    # --------------------------------------------------------

    Sxx = Sxx / np.max(Sxx)

    # ========================================================
    # SYSTEM FREQUENCY RESPONSE
    # ========================================================

    if system_type == 'Low-pass':

        # First-order low-pass:
        # H(z) = (1-alpha) / (1-alpha z^-1)

        H = (
            1.0 - alpha
        ) / (
            1.0
            - alpha * np.exp(-1j * omega)
        )

        system_name = f'Low-Pass, α = {alpha:.2f}'

    elif system_type == 'High-pass':

        # First-difference HP:
        # H(z) = 1 - z^-1

        H = (
            1.0
            - np.exp(-1j * omega)
        )

        system_name = 'First-Difference High-Pass'

    else:

        # Simple resonant band-pass-like response
        # Two-pole structure centered around +/- center*pi

        theta = center * np.pi

        r = alpha

        H = (
            1.0 - r
        ) / (
            1.0
            - 2.0 * r * np.cos(theta) * np.exp(-1j * omega)
            + r**2 * np.exp(-2j * omega)
        )

        system_name = f'Band-Pass, ω₀ = {center:.2f}π'

    # --------------------------------------------------------
    # SQUARED MAGNITUDE
    # --------------------------------------------------------

    H2 = np.abs(H) ** 2

    H2 = H2 / np.max(H2)

    # ========================================================
    # OUTPUT PSD
    # ========================================================

    Syy = Sxx * H2

    if np.max(Syy) > 0:

        Syy = Syy / np.max(Syy)

    # ========================================================
    # FIGURE
    # ========================================================

    fig, (ax1, ax2, ax3) = plt.subplots(
        1,
        3,
        figsize=(11.2, 3.8)
    )

    # ========================================================
    # GRAPH 1:
    # INPUT PSD
    # ========================================================

    ax1.plot(
        omega,
        Sxx,
        linewidth=2
    )

    ax1.set_xlim(
        -np.pi,
        np.pi
    )

    ax1.set_ylim(
        0,
        1.10
    )

    ax1.set_xticks(
        [
            -np.pi,
            -np.pi / 2,
            0,
            np.pi / 2,
            np.pi
        ]
    )

    ax1.set_xticklabels(
        [
            '-π',
            '-π/2',
            '0',
            'π/2',
            'π'
        ]
    )

    ax1.set_xlabel(
        'Angular frequency ω',
        fontsize=11
    )

    ax1.set_ylabel(
        'Normalized Sₓₓ(ω)',
        fontsize=11
    )

    ax1.set_title(
        f'Input PSD\n{input_name}',
        fontsize=12,
        pad=8
    )

    ax1.tick_params(
        axis='both',
        labelsize=9
    )

    ax1.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    # ========================================================
    # GRAPH 2:
    # |H|^2
    # ========================================================

    ax2.plot(
        omega,
        H2,
        linewidth=2
    )

    ax2.set_xlim(
        -np.pi,
        np.pi
    )

    ax2.set_ylim(
        0,
        1.10
    )

    ax2.set_xticks(
        [
            -np.pi,
            -np.pi / 2,
            0,
            np.pi / 2,
            np.pi
        ]
    )

    ax2.set_xticklabels(
        [
            '-π',
            '-π/2',
            '0',
            'π/2',
            'π'
        ]
    )

    ax2.set_xlabel(
        'Angular frequency ω',
        fontsize=11
    )

    ax2.set_ylabel(
        '|H(e^{jω})|²',
        fontsize=11
    )

    ax2.set_title(
        f'System Power Response\n{system_name}',
        fontsize=12,
        pad=8
    )

    ax2.tick_params(
        axis='both',
        labelsize=9
    )

    ax2.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    # ========================================================
    # GRAPH 3:
    # OUTPUT PSD
    # ========================================================

    ax3.plot(
        omega,
        Syy,
        linewidth=2
    )

    ax3.set_xlim(
        -np.pi,
        np.pi
    )

    ax3.set_ylim(
        0,
        1.10
    )

    ax3.set_xticks(
        [
            -np.pi,
            -np.pi / 2,
            0,
            np.pi / 2,
            np.pi
        ]
    )

    ax3.set_xticklabels(
        [
            '-π',
            '-π/2',
            '0',
            'π/2',
            'π'
        ]
    )

    ax3.set_xlabel(
        'Angular frequency ω',
        fontsize=11
    )

    ax3.set_ylabel(
        'Normalized Sᵧᵧ(ω)',
        fontsize=11
    )

    ax3.set_title(
        'Output PSD',
        fontsize=12,
        pad=8
    )

    ax3.tick_params(
        axis='both',
        labelsize=9
    )

    ax3.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    # --------------------------------------------------------
    # FIGURE SPACING
    # --------------------------------------------------------

    plt.subplots_adjust(
        left=0.06,
        right=0.98,
        top=0.85,
        bottom=0.20,
        wspace=0.30
    )

    plt.show()

# ============================================================
# RADIO BUTTONS
# ============================================================

input_selector = RadioButtons(
    options=[
        'White noise',
        'Colored AR(1)'
    ],
    value='White noise',
    description='',
    layout=Layout(width='240px')
)

input_selector.add_class(
    'horizontal-radio'
)

system_selector = RadioButtons(
    options=[
        'Low-pass',
        'High-pass',
        'Band-pass'
    ],
    value='Low-pass',
    description='',
    layout=Layout(width='300px')
)

system_selector.add_class(
    'horizontal-radio'
)

# ============================================================
# SLIDERS
# ============================================================

slider_style = {
    'description_width': '0px'
}

slider_layout = Layout(
    width='160px'
)

rho_slider = FloatSlider(
    min=0.0,
    max=0.98,
    step=0.02,
    value=0.85,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout,
    disabled=True
)

alpha_slider = FloatSlider(
    min=0.10,
    max=0.95,
    step=0.05,
    value=0.80,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

center_slider = FloatSlider(
    min=0.10,
    max=0.90,
    step=0.05,
    value=0.45,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout,
    disabled=True
)

# ============================================================
# CURRENT VALUE LABELS
# ============================================================

rho_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.85</div>'
)

alpha_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.80</div>'
)

center_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.45π</div>'
)

empty_value_1 = HTML(
    '<div></div>'
)

empty_value_2 = HTML(
    '<div></div>'
)

# ============================================================
# UPDATE CURRENT VALUE LABELS
# ============================================================

def update_rho_value(change):

    rho_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{rho_slider.value:.2f}</div>'

def update_alpha_value(change):

    alpha_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{alpha_slider.value:.2f}</div>'

def update_center_value(change):

    center_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{center_slider.value:.2f}π</div>'

rho_slider.observe(
    update_rho_value,
    names='value'
)

alpha_slider.observe(
    update_alpha_value,
    names='value'
)

center_slider.observe(
    update_center_value,
    names='value'
)

# ============================================================
# ENABLE / DISABLE CONTROLS
# ============================================================

def update_controls(change):

    rho_slider.disabled = (
        input_selector.value != 'Colored AR(1)'
    )

    if system_selector.value == 'High-pass':

        alpha_slider.disabled = True

        center_slider.disabled = True

    elif system_selector.value == 'Low-pass':

        alpha_slider.disabled = False

        center_slider.disabled = True

    else:

        alpha_slider.disabled = False

        center_slider.disabled = False

input_selector.observe(
    update_controls,
    names='value'
)

system_selector.observe(
    update_controls,
    names='value'
)

# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(
    plot_psd_transformation,
    input_type=input_selector,
    system_type=system_selector,
    rho=rho_slider,
    alpha=alpha_slider,
    center=center_slider
)

# ============================================================
# DOCUMENTATION
# ============================================================

theory_html = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.40;
    width:1050px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#12388c;
    margin-bottom:8px;
">
Transformation of Power Spectral Density Through an LTI System
</div>

<div style="margin-bottom:5px;">
<b>Input PSD:</b> Sₓₓ(ω) describes how the average power of the WSS input process is distributed over frequency.
</div>

<div style="margin-bottom:5px;">
<b>System:</b> the LTI system modifies each frequency component according to its squared magnitude response |H(e<sup>jω</sup>)|².
</div>

<div style="margin-bottom:5px;">
<b>Output PSD:</b> the output power spectrum is obtained from
</div>

<div style="
    font-family:serif;
    font-size:22px;
    font-style:italic;
    color:#12388c;
    margin:7px 0px 7px 20px;
">
Sᵧᵧ(ω) = |H(e<sup>jω</sup>)|² Sₓₓ(ω)
</div>

<div>
<b>This notebook:</b> visualizes the transformation as the spectral flow
Sₓₓ(ω) → |H(e<sup>jω</sup>)|² → Sᵧᵧ(ω).
</div>

</div>
""")

# ============================================================
# CONTROL LABELS
# ============================================================

input_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold;">Input:</div>'
)

system_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold;">System:</div>'
)

rho_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold;">AR parameter ρ:</div>'
)

alpha_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold;">System parameter α:</div>'
)

center_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold;">BP center ω₀:</div>'
)

# ============================================================
# CONTROL GRID
# ============================================================

controls_grid = GridBox(
    children=[
        input_label, input_selector, empty_value_1,
        system_label, system_selector, empty_value_2,
        rho_label, rho_slider, rho_value,
        alpha_label, alpha_slider, alpha_value,
        center_label, center_slider, center_value
    ],
    layout=Layout(
        width='670px',
        grid_template_columns='145px 300px 80px',
        grid_template_rows='34px 34px 34px 34px 34px',
        grid_gap='5px 8px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# CONTROLS CARD
# ============================================================

controls_card = VBox(
    [
        controls_grid
    ],
    layout=Layout(
        width='700px',
        min_width='700px',
        padding='12px 14px',
        border='1px solid #d2d2d2',
        margin='10px 0px 0px 0px',
        overflow='hidden'
    )
)

# ============================================================
# TOP AREA
# ============================================================

top_area = VBox(
    [
        theory_html,
        controls_card
    ],
    layout=Layout(
        width='1050px',
        overflow='hidden'
    )
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation_html = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.40;
    width:1050px;
    padding:12px 16px;
    border:1px solid #c8dfce;
    background:#f8fcf9;
    box-sizing:border-box;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#197b35;
    margin-bottom:7px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:5px;">
<b>Left:</b> the first graph is the power distribution already present in the input process.
</div>

<div style="margin-bottom:5px;">
<b>Center:</b> |H(e<sup>jω</sup>)|² determines which frequency regions are preserved or suppressed by the system.
</div>

<div style="margin-bottom:5px;">
<b>Right:</b> the output PSD is the point-by-point product of the first two curves.
</div>

<div style="
    margin-top:8px;
    font-family:serif;
    font-size:20px;
    font-style:italic;
    color:#197b35;
">
Sᵧᵧ(ω) = |H(e<sup>jω</sup>)|² Sₓₓ(ω)
</div>

</div>
""")

# ============================================================
# COMPLETE LAYOUT
# ============================================================

main_layout = VBox(
    [
        top_area,
        widget_plot.children[-1],
        interpretation_html
    ],
    layout=Layout(
        width='1050px',
        overflow='hidden'
    )
)

display(main_layout)